# Notebook 01 — 2D Lists & Nearest-Neighbor Interpolation

**Session:** Interpolation Practice Lab  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Master foundational 2D list manipulation (row-wise sums and matrix transposition) and build **Nearest-Neighbor Interpolation (NNI)** from 1D lists to horizontal, vertical, and full 2D grids (Problems 1–6).


## Problem 1 — Row-Wise Sum of a 2D List (`compute_sum`)


In [ ]:
def compute_sum(data: list[list[int | float]]) -> list[int | float]:
    """Return a list containing the sum of each row in a 2D list."""
    return [sum(row) for row in data]


assert compute_sum([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [6, 15, 24]
assert compute_sum([[1, 2], [3, 4], [5, 6]]) == [3, 7, 11]
print("Problem 1 passed:", compute_sum([[6, 8], [3, 1], [7, 3]]))

## Problem 2 — Matrix Transpose (`transpose`)


In [ ]:
def transpose(data: list[list[int | float]]) -> list[list[int | float]]:
    """Swap rows and columns of an m x n 2D list into an n x m 2D list."""
    if not data:
        return []
    rows = len(data)
    cols = len(data[0])
    return [[data[i][j] for i in range(rows)] for j in range(cols)]


assert transpose([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [
    [1, 4, 7],
    [2, 5, 8],
    [3, 6, 9],
]
assert transpose([[1, 2], [4, 5]]) == [[1, 4], [2, 5]]
assert transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]
print("Problem 2 passed:", transpose([[7, 2], [3, 5]]))

## Problem 3 — 1D Nearest-Neighbor Interpolation (`nni_1d`)

Given a 1D list of length $n$ and integer upsampling factor $k$, compute the scale ratio $s = n / (n \cdot k) = 1 / k$ and map each target index $i \in [0, nk - 1]$ back to $\lfloor i \cdot s \rfloor$ (or `i // k` in exact integer arithmetic).


In [ ]:
import math


def nni_1d(data: list[int | float], k: int) -> list[int | float]:
    """Upsample a 1D list by integer factor k using Nearest-Neighbor Interpolation."""
    if k <= 0:
        raise ValueError("Scale factor k must be a positive integer.")
    n = len(data)
    new_n = n * k
    scale = n / new_n
    return [data[min(n - 1, math.floor(i * scale))] for i in range(new_n)]


assert nni_1d([1, 2, 3], 2) == [1, 1, 2, 2, 3, 3]
assert nni_1d([1, 2, 3], 3) == [1, 1, 1, 2, 2, 2, 3, 3, 3]
print("Problem 3 passed:", nni_1d([5, 7], 3))

## Problem 4 — 2D Horizontal Nearest-Neighbor Interpolation (`h_nni_2d`)


In [ ]:
def h_nni_2d_index(data: list[list[int | float]], k: int) -> list[list[int | float]]:
    """Expand an m x n grid to m x (n * k) using explicit column-index mapping."""
    rows = len(data)
    cols = len(data[0])
    new_cols = cols * k
    w_scale = cols / new_cols
    return [
        [data[i][min(cols - 1, math.floor(j * w_scale))] for j in range(new_cols)]
        for i in range(rows)
    ]


def h_nni_2d(data: list[list[int | float]], k: int) -> list[list[int | float]]:
    """Simpler separable approach: apply nni_1d independently to each row."""
    return [nni_1d(row, k) for row in data]


grid_3x3 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
expected_3x6 = [
    [1, 1, 2, 2, 3, 3],
    [4, 4, 5, 5, 6, 6],
    [7, 7, 8, 8, 9, 9],
]
assert h_nni_2d_index(grid_3x3, 2) == expected_3x6
assert h_nni_2d(grid_3x3, 2) == expected_3x6
assert h_nni_2d([[1, 2], [4, 5], [7, 8]], 2) == [[1, 1, 2, 2], [4, 4, 5, 5], [7, 7, 8, 8]]
print("Problem 4 passed:", h_nni_2d([[3], [4]], 3))

## Problem 5 — 2D Vertical Nearest-Neighbor Interpolation (`v_nni_2d`)


In [ ]:
def v_nni_2d_index(data: list[list[int | float]], k: int) -> list[list[int | float]]:
    """Expand an m x n grid to (m * k) x n using explicit row-index mapping."""
    rows = len(data)
    cols = len(data[0])
    new_rows = rows * k
    h_scale = rows / new_rows
    return [
        [data[min(rows - 1, math.floor(i * h_scale))][j] for j in range(cols)]
        for i in range(new_rows)
    ]


def v_nni_2d(data: list[list[int | float]], k: int) -> list[list[int | float]]:
    """Simpler approach: duplicate each row k times vertically."""
    return [row[:] for row in data for _ in range(k)]


expected_6x3 = [
    [1, 2, 3],
    [1, 2, 3],
    [4, 5, 6],
    [4, 5, 6],
    [7, 8, 9],
    [7, 8, 9],
]
assert v_nni_2d_index(grid_3x3, 2) == expected_6x3
assert v_nni_2d(grid_3x3, 2) == expected_6x3
print("Problem 5 passed.")

## Problem 6 — Full 2D Nearest-Neighbor Interpolation (`nni_2d`)


In [ ]:
def nni_2d_direct(data: list[list[int | float]], k: int) -> list[list[int | float]]:
    """One-shot 2D NNI mapping target (i, j) directly to source (floor(i * s), floor(j * s))."""
    rows = len(data)
    cols = len(data[0])
    new_rows, new_cols = rows * k, cols * k
    h_scale, w_scale = rows / new_rows, cols / new_cols
    return [
        [
            data[min(rows - 1, math.floor(i * h_scale))][min(cols - 1, math.floor(j * w_scale))]
            for j in range(new_cols)
        ]
        for i in range(new_rows)
    ]


def nni_2d(data: list[list[int | float]], k: int) -> list[list[int | float]]:
    """Separable 2D NNI: horizontal pass followed by vertical pass."""
    return v_nni_2d(h_nni_2d(data, k), k)


expected_6x6 = [
    [1, 1, 2, 2, 3, 3],
    [1, 1, 2, 2, 3, 3],
    [4, 4, 5, 5, 6, 6],
    [4, 4, 5, 5, 6, 6],
    [7, 7, 8, 8, 9, 9],
    [7, 7, 8, 8, 9, 9],
]
assert nni_2d_direct(grid_3x3, 2) == expected_6x6
assert nni_2d(grid_3x3, 2) == expected_6x6
assert nni_2d([[1, 2], [4, 5], [7, 8]], 2) == [
    [1, 1, 2, 2],
    [1, 1, 2, 2],
    [4, 4, 5, 5],
    [4, 4, 5, 5],
    [7, 7, 8, 8],
    [7, 7, 8, 8],
]
print("Problem 6 passed: separable and one-shot 2D NNI match identically.")